[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/23_cross_attention_solution.ipynb)

# 🟠 Solution: Multi-Head Cross-Attention（参考版）

## 📋 题目描述

**难度：** Medium

实现多头交叉注意力（nn.Module）。

交叉注意力让解码器关注编码器输出：Q 来自一个序列，K/V 来自另一个序列，不使用因果掩码。

**签名:** `MultiHeadCrossAttention(d_model, num_heads)`（nn.Module）

**前向传播:** `forward(x_q, x_kv) -> Tensor`
- `x_q` — 查询输入 (B, S_q, d_model)
- `x_kv` — 键/值输入 (B, S_kv, d_model)

**返回:** 注意力输出 (B, S_q, d_model)

**约束:**
- 使用独立的 W_q、W_k、W_v、W_o 线性投影
- Q 和 KV 可以有不同的序列长度

**提示：** Q 来自 `x_q`，K/V 来自 `x_kv`。投影 → reshape 为 `(B, H, S, d_k)` → 缩放点积（无因果遮蔽）→ 拼接各头 → `W_o`。


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # ---- Step 1: 初始化参数 ----
        # num_heads: 注意力头数，比如 8
        # d_k: 每个头的维度 = d_model / num_heads
        # 例如 d_model=512, num_heads=8 → d_k=64
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # ---- Step 2: 定义四个投影矩阵 ----
        # W_q: 将 x_q 投影为 Query（查询向量）
        # W_k, W_v: 将 x_kv 投影为 Key 和 Value
        # W_o: 将多头输出合并后投影回 d_model
        # 注意：Q 来自解码器输入，K/V 来自编码器输出
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x_q, x_kv):
        B, S_q, _ = x_q.shape
        S_kv = x_kv.shape[1]

        # ---- Step 3: 线性投影 ----
        # x_q → Q: (B, S_q, d_model)
        # x_kv → K, V: (B, S_kv, d_model)
        q = self.W_q(x_q)
        k = self.W_k(x_kv)
        v = self.W_v(x_kv)

        # ---- Step 4: 拆分多头 ----
        # (B, S, d_model) → (B, S, num_heads, d_k) → (B, num_heads, S, d_k)
        # .view 拆分最后一维为 (num_heads, d_k)
        # .transpose(1,2) 把 num_heads 维提前，方便批量矩阵乘法
        q = q.view(B, S_q, self.num_heads, self.d_k).transpose(1, 2)
        k = k.view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)
        v = v.view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)
        # 现在 shape: (B, num_heads, S, d_k)

        # ---- Step 5: 缩放点积注意力 ----
        # scores = Q @ K^T / sqrt(d_k)
        # (B,H,S_q,d_k) @ (B,H,d_k,S_kv) → (B,H,S_q,S_kv)
        # 除以 sqrt(d_k) 防止点积值过大导致 softmax 饱和
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

        # ---- Step 6: Softmax 归一化 ----
        # 沿最后一个维度（S_kv）做 softmax
        # 每个 query 对所有 key 的注意力权重，和为 1
        # 注意：交叉注意力没有因果掩码！Q 可以关注所有 KV
        weights = torch.softmax(scores, dim=-1)

        # ---- Step 7: 加权求和 ----
        # (B,H,S_q,S_kv) @ (B,H,S_kv,d_k) → (B,H,S_q,d_k)
        # 每个 query 位置得到所有 value 的加权组合
        attn = torch.matmul(weights, v)

        # ---- Step 8: 合并多头并投影 ----
        # transpose 回 (B, S_q, num_heads, d_k)
        # .contiguous() 确保内存连续（transpose 后可能不连续）
        # .view 拼接为 (B, S_q, d_model)
        # W_o 投影回 d_model 维度
        return self.W_o(attn.transpose(1, 2).contiguous().view(B, S_q, -1))


In [ ]:
mha = MultiHeadCrossAttention(d_model=64, num_heads=8)
x_q = torch.randn(2, 10, 64)   # decoder query: 10 tokens
x_kv = torch.randn(2, 20, 64)  # encoder kv: 20 tokens
print('Output:', mha(x_q, x_kv).shape)


In [ ]:
from torch_judge import check
check('cross_attention')


## 📝 核心思路总结

1. **Q/K/V 分离**：Q 来自解码器，K/V 来自编码器，各自独立投影
2. **多头拆分**：`view + transpose` 将 d_model 拆为 num_heads × d_k
3. **缩放点积**：`QK^T / sqrt(d_k)` + softmax + 加权 V
4. **无因果掩码**：交叉注意力允许 Q 关注所有 KV 位置
